# One dataset, five conformal methods

This compact tutorial compares every estimator on the same heteroscedastic candy dataset. It focuses on the shared **fit → conformalize → evaluate** workflow rather than listing every method on every class.

## Generate the candy data

The noise scale narrows, widens, then narrows across the input range. A useful adaptive region should follow that shape.

In [1]:
import numpy as np
import pandas as pd
import torch
import zuko
from sklearn.ensemble import HistGradientBoostingRegressor

from pitcp import CONTRA, CQR, HPD, PITCP, SCP


def _scale(x):
    return np.abs(1 - 2 * x**2) + 0.1


rng = np.random.RandomState(42)


def _sample(n):
    x = rng.rand(n) * 2 - 1
    return x[:, None], rng.randn(n) * _scale(x)


(X_train, y_train), (X_cal, y_cal), (X_test, y_test) = [
    _sample(n) for n in (1000, 300, 1000)
]
level = 0.9

## Fit PITCP and the split-conformal baseline

Both methods use the absolute response as their scalar nonconformity score. PITCP learns its conditional distribution; SCP uses one global threshold.

In [2]:
torch.manual_seed(42)
score_flow = zuko.flows.SOSPF(features=1, context=1, hidden_features=(16, 16))
pitcp = PITCP(
    score_flow,
    torch.optim.Adam(score_flow.parameters(), lr=1e-3),
    n_epochs=20,
    batch_size=256,
    verbose=False,
    random_state=42,
)
pitcp.fit(X_train, np.abs(y_train))
pitcp.conformalize(X_cal, np.abs(y_cal))

scp = SCP()
scp.conformalize(np.abs(y_cal));

## Fit CQR at the requested confidence level

CQR learns the lower and upper response quantiles themselves, so changing `level` requires a new fit.

In [3]:
cqr = CQR(
    HistGradientBoostingRegressor(random_state=42),
    confidence_level=level,
)
cqr.fit(X_train, y_train)
cqr.conformalize(X_cal, y_cal);

## Reuse one response density for HPD and CONTRA

HPD calibrates density ranks, while CONTRA calibrates distances in the flow's latent space. The fitted conditional flow can be shared.

In [4]:
torch.manual_seed(42)
response_flow = zuko.flows.MAF(features=1, context=1, hidden_features=(16, 16))
optimizer = torch.optim.Adam(response_flow.parameters(), lr=1e-3)
hpd = HPD(
    response_flow,
    optimizer,
    n_epochs=20,
    n_samples=256,
    batch_size=256,
    verbose=False,
    random_state=42,
)
hpd.fit(X_train, y_train)
hpd.conformalize(X_cal, y_cal);

contra = CONTRA(response_flow, optimizer, batch_size=256, verbose=False)
contra.conformalize(X_cal, y_cal);

## Compare marginal coverage

Every estimator exposes `contains`; only the arguments differ according to whether its region is defined in score space or response space.

In [5]:
covered = {
    "PITCP": pitcp.contains(X_test, np.abs(y_test), confidence_level=level),
    "SCP": scp.contains(np.abs(y_test), confidence_level=level),
    "CQR": cqr.contains(X_test, y_test),
    "HPD": hpd.contains(X_test, y_test, confidence_level=level),
    "CONTRA": contra.contains(X_test, y_test, confidence_level=level),
}
pd.DataFrame(
    {"Coverage": {name: values.mean() for name, values in covered.items()}}
).style.format("{:.1%}")

,Coverage
PITCP,90.7%
SCP,90.8%
CQR,89.1%
HPD,90.5%
CONTRA,89.0%


## Use the utility functions

The utilities summarize conditional behavior and region size without changing the fitted estimators.

In [6]:
from pitcp.utils import (
    contra_volume,
    coverage_gap,
    cqr_volume,
    hpd_volume,
    lp_volume,
)

groups = np.digitize(X_test[:, 0], [-0.5, 0, 0.5])
utility_summary = pd.DataFrame(
    {
        "Metric": [
            "PITCP coverage gap",
            "SCP mean volume",
            "CQR mean volume",
            "HPD mean volume",
            "CONTRA mean volume",
        ],
        "Value": [
            coverage_gap(groups, covered["PITCP"]),
            lp_volume(scp, X_test, d=1, confidence_level=level).mean(),
            cqr_volume(cqr, X_test).mean(),
            hpd_volume(hpd, X_test, level, n_samples=128).mean(),
            contra_volume(contra, X_test, level, n_samples=128).mean(),
        ],
    }
)
utility_summary.style.format({"Value": "{:.3f}"}).hide(axis="index")

Metric,Value
PITCP coverage gap,0.107
SCP mean volume,2.702
CQR mean volume,2.506
HPD mean volume,2.639
CONTRA mean volume,2.514


The notebook uses short training runs so it remains easy to execute. The interactive playground uses the paper-sized 5,000/1,000/5,000 split and pretrained density models; try it next to inspect the full prediction regions.